# 🧠 Logic Fine-Tuning: Qwen3-8B × Unsloth
**Pipeline:** Data Unpacking → LoRA Training → Chain-of-Thought Alignment → Evaluation

> Dataset: `Logic_Based_Educational_Queries-2.json` | 411 indices | FOL + NL premises  
> Target: **≥60% accuracy** trên toàn bộ test set

---
## 📋 Fixes so với phiên bản cũ
| # | Vấn đề cũ | Fix mới |
|---|-----------|---------|
| 1 | Qwen2.5-7B | **Qwen3-8B** (mạnh hơn về reasoning) |
| 2 | `formatting_prompts_func` định nghĩa 2 lần | Chỉ 1 lần, đúng signature |
| 3 | `train_on_responses_only` dùng sai token markers | Dùng đúng `<|im_start|>` cho Qwen3 |
| 4 | `dataset_text_field="text"` nhưng `formatting_func` trả list | Fix để SFTTrainer nhận đúng |
| 5 | LR=1e-5 quá thấp (under-fit) | **LR=2e-4** (chuẩn LoRA) |
| 6 | Evaluation dùng raw_data[-3:] không đúng split | Fix dùng đúng test split |
| 7 | `extract_final_answer` regex yếu | Regex mạnh hơn + fallback |
| 8 | Inference thiếu `repetition_penalty` | Thêm penalty để tránh loop |
| 9 | Evaluate chỉ trên vài sample | **Evaluate toàn bộ 411 indices** |

In [1]:
%%capture
# Unsloth với CUDA 12.1 (T4 / A100 Kaggle)
!pip install unsloth
!pip install --upgrade --no-cache-dir trl peft accelerate bitsandbytes
!pip install datasets transformers sentencepiece protobuf

In [2]:
import json
import os
import re
import random
import copy
from collections import Counter
from typing import List, Dict, Any, Optional
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# Seed cố định để reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ PyTorch version: 2.10.0+cu128
✅ CUDA available: True
✅ GPU: Tesla T4
✅ VRAM: 15.6 GB


In [3]:
# ============================================================
#  GLOBAL CONFIG
# ============================================================

# --- Path ---
DATA_PATH  = "/kaggle/input/datasets/thurdayafternoon/logic2026/Logic_Based_Educational_Queries-2.json"
OUTPUT_DIR = "/kaggle/working/lora_output_qwen3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Model (FIX #1: Qwen3-8B thay vì Qwen2.5-7B) ---
MODEL_NAME  = "unsloth/Qwen3-8B-bnb-4bit"   # 4-bit quantized, tối ưu T4/A100
MAX_SEQ_LEN = 2048

# --- LoRA Config ---
LORA_R       = 64      # Rank cao hơn để học logic sâu hơn
LORA_ALPHA   = 128      # = 2 * LORA_R
LORA_DROPOUT = 0.05

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# --- Training Hyperparams (FIX #5: LR chuẩn LoRA) ---
LEARNING_RATE    = 1e-4     # Chuẩn LoRA; 1e-5 cũ gây under-fit
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 8        # Effective batch = 16
NUM_EPOCHS       = 4
WARMUP_RATIO     = 0.05
LR_SCHEDULER     = "cosine"
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 1.0

# --- Data Split ---
TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05

# --- Adversarial Weighting ---
UNKNOWN_OVERSAMPLE_FACTOR = 3

print("✅ Config loaded.")
print(f"   Model         : {MODEL_NAME}")
print(f"   LoRA rank     : {LORA_R}, alpha: {LORA_ALPHA}")
print(f"   Effective BS  : {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"   Learning rate : {LEARNING_RATE}")
print(f"   Epochs        : {NUM_EPOCHS}")

✅ Config loaded.
   Model         : unsloth/Qwen3-8B-bnb-4bit
   LoRA rank     : 64, alpha: 128
   Effective BS  : 16
   Learning rate : 0.0001
   Epochs        : 4


In [4]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data: List[Dict] = json.load(f)

print(f"📦 Tổng số indices: {len(raw_data)}")
print("\n--- Cấu trúc 1 index mẫu ---")
sample = raw_data[0]
print(f"  idx            : {sample['idx']}")
print(f"  premises-FOL   : {len(sample['premises-FOL'])} premises")
print(f"  premises-NL    : {len(sample['premises-NL'])} premises")
print(f"  questions      : {len(sample['questions'])} questions")
print(f"  answers        : {sample['answers']}")

all_answers = []
for item in raw_data:
    all_answers.extend(item["answers"])

answer_dist = Counter(all_answers)
total_ans = sum(answer_dist.values())
print(f"\n📊 Phân phối nhãn (tổng {total_ans} mẫu sau unpack):")
for label, count in sorted(answer_dist.items()):
    pct = count / total_ans * 100
    bar = "█" * int(pct / 2)
    print(f"  {label:10s}: {count:4d} ({pct:5.1f}%) {bar}")

📦 Tổng số indices: 411

--- Cấu trúc 1 index mẫu ---
  idx            : [[6, 8, 12, 16], [2, 4, 5, 15]]
  premises-FOL   : 16 premises
  premises-NL    : 16 premises
  questions      : 2 questions
  answers        : ['Unknown', 'Yes']

📊 Phân phối nhãn (tổng 812 mẫu sau unpack):
  A         :  161 ( 19.8%) █████████
  B         :   43 (  5.3%) ██
  C         :   37 (  4.6%) ██
  D         :   22 (  2.7%) █
  No        :  156 ( 19.2%) █████████
  Unknown   :  150 ( 18.5%) █████████
  Yes       :  243 ( 29.9%) ██████████████


In [5]:
SYSTEM_PROMPT = """You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).
Given a set of premises (in natural language and FOL notation), you must:
1. Analyze the logical structure of each premise carefully.
2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation, existential instantiation, hypothetical syllogism.
3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.
4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.
5. For Yes/No questions: verify the statement logically.
6. If the premises are INSUFFICIENT to determine a unique conclusion, your Final Answer MUST be exactly: Unknown
7. Never hallucinate a conclusion that is not entailed by the premises.
Format your response EXACTLY as:
### Step-by-Step Reasoning
<your reasoning here>
### Final Answer
<A or B or C or D or Yes or No or Unknown>"""


def format_premises(fol_list: List[str], nl_list: List[str]) -> str:
    lines = ["### Premises"]
    for i, (fol, nl) in enumerate(zip(fol_list, nl_list), 1):
        lines.append(f"P{i}. [NL]  {nl}")
        lines.append(f"   [FOL] {fol}")
    return "\n".join(lines)


def build_user_message(fol_list: List[str], nl_list: List[str], question: str) -> str:
    premises_block = format_premises(fol_list, nl_list)
    return f"{premises_block}\n\n### Question\n{question}"


def build_assistant_message(explanation: str, answer: str) -> str:
    """Answer-Last: CoT trước, đáp án sau để model học suy luận."""
    return (
        f"### Step-by-Step Reasoning\n{explanation}\n\n"
        f"### Final Answer\n{answer}"
    )


print("✅ Prompt templates defined.")
s = raw_data[0]
print("\n--- User Message preview ---")
print(build_user_message(s["premises-FOL"], s["premises-NL"], s["questions"][0])[:400])
print("\n--- Assistant Message preview ---")
print(build_assistant_message(s["explanation"][0], s["answers"][0])[:300])

✅ Prompt templates defined.

--- User Message preview ---
### Premises
P1. [NL]  All students must enroll in at least one core subject.
   [FOL] ∀x (EnrollsCoreSubject(x))
P2. [NL]  If a student attends all lectures, they will have a higher chance of passing the course.
   [FOL] ∀x (AttendsLectures(x) → HigherChancePass(x))
P3. [NL]  If a student does not attend all lectures, they might struggle to understand key concepts.
   [FOL] ∀x (¬AttendsLectures(x

--- Assistant Message preview ---
### Step-by-Step Reasoning
From premise 6 (There exists at least one student who participates in an academic competition), premise 8 (If a student joins the research group, they must contribute to at least one published paper), premise 12 (If joining the research group requires contributing to a pub


In [6]:
def unpack_dataset(raw_data: List[Dict]) -> List[Dict]:
    """Mỗi (index × question) → 1 training sample độc lập."""
    samples = []
    for entry in raw_data:
        fol_list     = entry["premises-FOL"]
        nl_list      = entry["premises-NL"]
        questions    = entry["questions"]
        answers      = entry["answers"]
        explanations = entry["explanation"]

        assert len(questions) == len(answers) == len(explanations), (
            f"Mismatch in entry idx={entry.get('idx')}: "
            f"q={len(questions)}, a={len(answers)}, exp={len(explanations)}"
        )

        for q, a, exp in zip(questions, answers, explanations):
            user_msg      = build_user_message(fol_list, nl_list, q)
            assistant_msg = build_assistant_message(exp, a)
            sample = {
                "messages": [
                    {"role": "system",    "content": SYSTEM_PROMPT},
                    {"role": "user",      "content": user_msg},
                    {"role": "assistant", "content": assistant_msg},
                ],
                "answer_label": a,
                "is_unknown":   a.strip().lower() == "unknown",
                # Lưu gốc để evaluate toàn bộ 411 indices sau
                "_raw_entry_idx": id(entry),
            }
            samples.append(sample)
    return samples


flat_samples = unpack_dataset(raw_data)
unknown_count = sum(1 for s in flat_samples if s["is_unknown"])
print(f"📦 Tổng samples sau unpack  : {len(flat_samples)}")
print(f"   Nhãn Unknown             : {unknown_count} ({unknown_count/len(flat_samples)*100:.1f}%)")
print(f"   Nhãn có đáp án           : {len(flat_samples) - unknown_count}")

📦 Tổng samples sau unpack  : 812
   Nhãn Unknown             : 150 (18.5%)
   Nhãn có đáp án           : 662


In [7]:
def apply_adversarial_oversampling(
    samples: List[Dict],
    factor: int = UNKNOWN_OVERSAMPLE_FACTOR,
    seed: int = SEED
) -> List[Dict]:
    rng = random.Random(seed)
    unknown_samples = [s for s in samples if s["is_unknown"]]
    known_samples   = [s for s in samples if not s["is_unknown"]]

    augmented_unknowns = unknown_samples * factor
    combined = known_samples + augmented_unknowns
    rng.shuffle(combined)

    new_unknown_count = sum(1 for s in combined if s["is_unknown"])
    print(f"✅ Sau Adversarial Oversampling (factor={factor}x):")
    print(f"   Tổng samples  : {len(combined)}")
    print(f"   Unknown       : {new_unknown_count} ({new_unknown_count/len(combined)*100:.1f}%)")
    print(f"   Known answers : {len(combined) - new_unknown_count}")
    return combined


augmented_samples = apply_adversarial_oversampling(flat_samples)

✅ Sau Adversarial Oversampling (factor=3x):
   Tổng samples  : 1112
   Unknown       : 450 (40.5%)
   Known answers : 662


In [8]:
def split_dataset(
    samples: List[Dict],
    train_ratio: float = TRAIN_RATIO,
    val_ratio:   float = VAL_RATIO,
    seed: int = SEED,
) -> DatasetDict:
    shuffled = copy.deepcopy(samples)
    random.Random(seed).shuffle(shuffled)

    n = len(shuffled)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    train_list = shuffled[:n_train]
    val_list   = shuffled[n_train : n_train + n_val]
    test_list  = shuffled[n_train + n_val :]

    def to_hf_dataset(lst):
        return Dataset.from_list([{"messages": s["messages"]} for s in lst])

    ds = DatasetDict({
        "train":      to_hf_dataset(train_list),
        "validation": to_hf_dataset(val_list),
        "test":       to_hf_dataset(test_list),
    })

    print("📊 Dataset splits:")
    for split_name, split_ds in ds.items():
        print(f"   {split_name:12s}: {len(split_ds):5d} samples")
    return ds


dataset = split_dataset(augmented_samples)

📊 Dataset splits:
   train       :   945 samples
   validation  :   111 samples
   test        :    56 samples


In [9]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

# FIX #3: Qwen3 dùng chat template "qwen-2.5" (tương thích Qwen3)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Vocab size    : {tokenizer.vocab_size}")
print(f"   Max seq length: {MAX_SEQ_LEN}")

==((====))==  Unsloth 2026.5.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.07G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-8B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model loaded: unsloth/Qwen3-8B-bnb-4bit
   Vocab size    : 151643
   Max seq length: 2048


In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = LORA_TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    use_rslora                 = False,
    loftq_config               = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied:")
print(f"   Trainable params: {trainable:,} ({trainable/total_p*100:.2f}%)")
print(f"   Total params    : {total_p:,}")
print(f"   Target modules  : {LORA_TARGET_MODULES}")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.8 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA applied:
   Trainable params: 174,587,904 (3.57%)
   Total params    : 4,892,439,552
   Target modules  : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


In [11]:
# FIX #2 & #4: Chỉ định nghĩa 1 lần, đúng signature cho SFTTrainer
# SFTConfig với dataset_text_field="text" yêu cầu formatting_func trả về dict
# Nhưng khi dùng formatting_func KHÔNG có dataset_text_field → trả list of str

def formatting_prompts_func(examples):
    """
    Trả về list of strings (không dict).
    SFTTrainer với formatting_func sẽ tự tạo cột 'text' từ list này.
    """
    convos = examples["messages"]
    # Unsloth có thể gửi single sample (list of dicts) thay vì batch
    if convos and isinstance(convos[0], dict):
        convos = [convos]

    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize              = False,
            add_generation_prompt = False,
        )
        for convo in convos
    ]
    return texts   # list of str, không phải dict


# Verify formatting
sample_text = formatting_prompts_func({"messages": dataset["train"][0]["messages"]})
print("--- Formatted sample (đầu 600 ký tự) ---")
if isinstance(sample_text, list):
    print(sample_text[0][:600])
else:
    print(sample_text[:600])
print("\n[... truncated ...]")

--- Formatted sample (đầu 600 ký tự) ---
<|im_start|>system
You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).
Given a set of premises (in natural language and FOL notation), you must:
1. Analyze the logical structure of each premise carefully.
2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation, existential instantiation, hypothetical syllogism.
3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.
4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.
5. For Yes/No questions: verify the statement logicall

[... truncated ...]


In [12]:
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm
import torch

# ── Custom Progress Callback ───────────────────────────────────────
class TqdmProgressCallback(TrainerCallback):
    def __init__(self):
        self.pbar = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="🚀 Training",
                         unit="step", colour="green")
    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.update(1)
            if state.log_history:
                self.pbar.set_postfix({"loss": f"{state.log_history[-1].get('loss', 0):.4f}"})
    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()

# ── Áp dụng Chat Template trước khi train để tránh lỗi Jinja2 ──
def apply_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return example

train_ds = dataset["train"].map(apply_template)
eval_ds  = dataset["validation"].map(apply_template)

_steps_per_epoch = max(1, len(train_ds) // (BATCH_SIZE * GRAD_ACCUM_STEPS))
_logging_steps   = max(1, min(10, _steps_per_epoch // 10))
# FIX #5: Không dùng dataset_text_field khi đã dùng formatting_func
# FIX #6: Thêm các tham số ổn định training
training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    # dataset_text_field KHÔNG set khi dùng formatting_func
    packing                     = False,
    padding_free                = False,
    assistant_only_loss         = False,  # Tắt để tránh xung đột với train_on_responses_only
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    num_train_epochs            = NUM_EPOCHS,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = MAX_GRAD_NORM,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 20,
    eval_strategy               = "steps",
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    seed                        = SEED,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    optim                       = "adamw_8bit",
    push_to_hub                 = False,
)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset["train"],
    eval_dataset     = dataset["validation"],
    formatting_func  = formatting_prompts_func,
    args             = training_args,
)

# FIX #3: Qwen3 tokens đúng cho train_on_responses_only
# Verify token markers trước khi dùng
sample_formatted = tokenizer.apply_chat_template(
    dataset["train"][0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print("Token markers check:")
print("  <|im_start|>user    present:", "<|im_start|>user" in sample_formatted)
print("  <|im_start|>assistant present:", "<|im_start|>assistant" in sample_formatted)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

print("\n✅ SFTTrainer initialized successfully!")
print(f"   Train samples : {len(dataset['train'])}")
print(f"   Val samples   : {len(dataset['validation'])}")

Map:   0%|          | 0/945 [00:00<?, ? examples/s]

Map:   0%|          | 0/111 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/945 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/111 [00:00<?, ? examples/s]

...

Map (num_proc=8):   0%|          | 0/945 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/945 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/111 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/111 [00:00<?, ? examples/s]


✅ SFTTrainer initialized successfully!
   Train samples : 945
   Val samples   : 111


In [13]:
print("🚀 Starting fine-tuning...")
print("=" * 60)

trainer_stats = trainer.train()

print("\n" + "=" * 60)
print("✅ Training complete!")
print(f"   Total steps    : {trainer_stats.global_step}")
print(f"   Training time  : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Samples/second : {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"   Final loss     : {trainer_stats.metrics['train_loss']:.4f}")

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"   Peak VRAM used : {peak:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 Starting fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 945 | Num Epochs = 4 | Total steps = 240
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 174,587,904 of 8,365,323,264 (2.09% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.090660,0.155592
200,0.016280,0.125070
240,0.020936,0.128152


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


✅ Training complete!
   Total steps    : 240
   Training time  : 14386s
   Samples/second : 0.26
   Final loss     : 0.1425
   Peak VRAM used : 10.87 GB


In [14]:
LORA_SAVE_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
os.makedirs(LORA_SAVE_PATH, exist_ok=True)

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

saved_files = os.listdir(LORA_SAVE_PATH)
total_size_mb = sum(
    os.path.getsize(os.path.join(LORA_SAVE_PATH, f))
    for f in saved_files
) / 1e6

print("✅ LoRA adapter saved.")
print(f"   Path  : {LORA_SAVE_PATH}")
print(f"   Files : {saved_files}")
print(f"   Size  : {total_size_mb:.1f} MB")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lora_output_qwen3/lora_adapter/tokenizer_config.json.


✅ LoRA adapter saved.
   Path  : /kaggle/working/lora_output_qwen3/lora_adapter
   Files : ['chat_template.jinja', 'tokenizer.json', 'adapter_config.json', 'tokenizer_config.json', 'adapter_model.safetensors', 'README.md']
   Size  : 709.9 MB


In [15]:
# Chuyển sang inference mode
FastLanguageModel.for_inference(model)


def predict(fol_list: List[str], nl_list: List[str], question: str,
            max_new_tokens: int = 512, temperature: float = 0.0) -> str:
    """Inference với repetition_penalty để tránh loop (FIX #8)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_message(fol_list, nl_list, question)},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens      = max_new_tokens,
            temperature         = temperature,
            do_sample           = temperature > 0,
            repetition_penalty  = 1.0,    # FIX #8: tránh vòng lặp token
            pad_token_id        = tokenizer.eos_token_id,
            eos_token_id        = tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("✅ predict() function ready with repetition_penalty=1.1")

✅ predict() function ready with repetition_penalty=1.1


In [16]:
def extract_final_answer(model_output: str) -> str:
    """
    FIX #7: Regex mạnh hơn — nhiều pattern fallback để không miss.
    Priority order: Final Answer block > inline answer patterns.
    """
    text = model_output.strip()

    # Pattern 1: phần sau '### Final Answer'
    match = re.search(
        r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)",
        text, re.IGNORECASE
    )
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        if re.match(r"^unknown", ans, re.IGNORECASE):
            return "Unknown"
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        # Yes/No
        if re.match(r"^yes", ans, re.IGNORECASE):
            return "Yes"
        if re.match(r"^no", ans, re.IGNORECASE):
            return "No"

    # Pattern 2: "Answer: X" hoặc "answer is X"
    match2 = re.search(
        r"(?:answer\s*(?:is|:)\s*)([A-D]|unknown|yes|no)",
        text, re.IGNORECASE
    )
    if match2:
        g = match2.group(1).strip()
        if g.lower() == "unknown": return "Unknown"
        if g.lower() == "yes":    return "Yes"
        if g.lower() == "no":     return "No"
        return g.upper()

    # Pattern 3: standalone letter near end of text
    last_500 = text[-500:]
    match3 = re.findall(r"\b([A-D])\b", last_500)
    if match3:
        return match3[-1].upper()

    # Pattern 4: Unknown/Yes/No anywhere
    if re.search(r"\bunknown\b", text, re.IGNORECASE):
        return "Unknown"
    if re.search(r"\byes\b", text, re.IGNORECASE):
        return "Yes"
    if re.search(r"\bno\b", text, re.IGNORECASE):
        return "No"

    return "UNPARSEABLE"


# Test
test_cases = [
    ("### Final Answer\nA", "A"),
    ("### Final Answer\nUnknown", "Unknown"),
    ("### Final Answer\nYes", "Yes"),
    ("some reasoning\n### Final Answer\nB. Because...", "B"),
    ("the answer is C", "C"),
    ("UNPARSEABLE output xyz", "UNPARSEABLE"),
]
print("Testing extract_final_answer:")
for text, expected in test_cases:
    got = extract_final_answer(text)
    status = "✅" if got == expected else "❌"
    print(f"  {status} '{text[:40]}'  →  got={got}, expected={expected}")

Testing extract_final_answer:
  ✅ '### Final Answer
A'  →  got=A, expected=A
  ✅ '### Final Answer
Unknown'  →  got=Unknown, expected=Unknown
  ✅ '### Final Answer
Yes'  →  got=Yes, expected=Yes
  ✅ 'some reasoning
### Final Answer
B. Becau'  →  got=B, expected=B
  ✅ 'the answer is C'  →  got=C, expected=C
  ✅ 'UNPARSEABLE output xyz'  →  got=UNPARSEABLE, expected=UNPARSEABLE


In [17]:
# FIX #9: Evaluate toàn bộ 411 indices (không chỉ vài sample cuối)
print("📊 Evaluating trên toàn bộ 411 indices...")
print("=" * 60)

correct       = 0
total_eval    = 0
unknown_correct = 0
unknown_total = 0
results_log   = []

for idx_i, entry in enumerate(raw_data):
    for q_i, (q, expected_ans, _) in enumerate(
        zip(entry["questions"], entry["answers"], entry["explanation"])
    ):
        output = predict(
            entry["premises-FOL"],
            entry["premises-NL"],
            q,
            max_new_tokens = 256,
            temperature    = 0.0,   # Greedy decoding → deterministic
        )
        pred       = extract_final_answer(output)
        is_correct = (pred.strip().lower() == expected_ans.strip().lower())

        total_eval += 1
        if is_correct:
            correct += 1

        if expected_ans.strip().lower() == "unknown":
            unknown_total += 1
            if is_correct:
                unknown_correct += 1

        results_log.append({
            "idx":      idx_i,
            "q_i":      q_i,
            "expected": expected_ans,
            "pred":     pred,
            "correct":  is_correct,
        })

        # Progress mỗi 50 entries
        if (idx_i + 1) % 50 == 0 and q_i == 0:
            running_acc = correct / total_eval * 100
            print(f"  [{idx_i+1:3d}/411] Running accuracy: {running_acc:.1f}%")

print("\n" + "=" * 60)
print("✅ FINAL EVALUATION RESULTS")
print("=" * 60)
overall_acc = correct / total_eval * 100 if total_eval > 0 else 0
print(f"Overall Accuracy  : {correct}/{total_eval} = {overall_acc:.1f}%")

if unknown_total > 0:
    unk_acc = unknown_correct / unknown_total * 100
    print(f"Unknown Accuracy  : {unknown_correct}/{unknown_total} = {unk_acc:.1f}%")

known_total   = total_eval - unknown_total
known_correct = correct - unknown_correct
if known_total > 0:
    known_acc = known_correct / known_total * 100
    print(f"Known Acc         : {known_correct}/{known_total} = {known_acc:.1f}%")

unparseable = sum(1 for r in results_log if r["pred"] == "UNPARSEABLE")
print(f"Unparseable preds : {unparseable}/{total_eval}")

if overall_acc >= 60:
    print(f"\n🎯 TARGET MET: {overall_acc:.1f}% ≥ 60% ✅")
else:
    print(f"\n⚠️  Target not met: {overall_acc:.1f}% < 60%")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


📊 Evaluating trên toàn bộ 411 indices...


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

  [ 50/411] Running accuracy: 86.6%


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

  [350/411] Running accuracy: 69.5%


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [400/411] Running accuracy: 70.0%


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


✅ FINAL EVALUATION RESULTS
Overall Accuracy  : 569/812 = 70.1%
Unknown Accuracy  : 138/150 = 92.0%
Known Acc         : 431/662 = 65.1%
Unparseable preds : 20/812

🎯 TARGET MET: 70.1% ≥ 60% ✅


In [18]:
# Phân tích lỗi để hiểu pattern
wrong_samples = [r for r in results_log if not r["correct"]]
print(f"\n🔍 Error Analysis ({len(wrong_samples)} wrong predictions)")
print("=" * 50)

# Confusion matrix
from collections import defaultdict
confusion = defaultdict(Counter)
for r in results_log:
    confusion[r["expected"]][r["pred"]] += 1

print("\nConfusion (expected → predicted):")
for expected_label in sorted(confusion.keys()):
    row = confusion[expected_label]
    total_row = sum(row.values())
    print(f"  {expected_label:8s} (n={total_row:3d}): ", end="")
    for pred_label, count in sorted(row.items()):
        print(f"{pred_label}={count}", end="  ")
    print()

# Mẫu sai đầu tiên
print(f"\n--- 3 mẫu sai đầu tiên ---")
for r in wrong_samples[:3]:
    entry = raw_data[r["idx"]]
    q     = entry["questions"][r["q_i"]]
    print(f"  idx={r['idx']}, q={r['q_i']}")
    print(f"  Q: {q[:80]}...")
    print(f"  Expected: {r['expected']} | Predicted: {r['pred']}")
    print()


🔍 Error Analysis (243 wrong predictions)

Confusion (expected → predicted):
  A        (n=161): A=101  B=6  C=4  UNPARSEABLE=3  Unknown=47  
  B        (n= 43): A=4  B=29  C=1  UNPARSEABLE=1  Unknown=8  
  C        (n= 37): C=18  Unknown=19  
  D        (n= 22): A=1  D=15  UNPARSEABLE=2  Unknown=4  
  No       (n=156): No=73  UNPARSEABLE=2  Unknown=4  Yes=77  
  Unknown  (n=150): C=2  D=2  No=1  UNPARSEABLE=6  Unknown=138  Yes=1  
  Yes      (n=243): No=40  UNPARSEABLE=6  Unknown=2  Yes=195  

--- 3 mẫu sai đầu tiên ---
  idx=1, q=0
  Q: Based on the premises, which of the following statements is correct?
A. If at le...
  Expected: A | Predicted: Unknown

  idx=4, q=1
  Q: Based on the above premises, is the statement true?
Statement: 'If a student doe...
  Expected: No | Predicted: Yes

  idx=11, q=1
  Q: Based on the premises, is the following statement true?
Statement: ∃x U(x)...
  Expected: No | Predicted: Yes



In [19]:
# Optional: Push lên HuggingFace Hub
PUSH_TO_HUB = False
HF_REPO_ID  = "Barakuga/qwen3-8b-logic-lora"

if PUSH_TO_HUB:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")

    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        print(f"✅ Pushed to: https://huggingface.co/{HF_REPO_ID}")
    else:
        print("⚠️  HF_TOKEN not found. Set via Kaggle Secrets.")
else:
    print("ℹ️  PUSH_TO_HUB=False. Đặt True để push lên HuggingFace Hub.")

ℹ️  PUSH_TO_HUB=False. Đặt True để push lên HuggingFace Hub.
